# Fashion Compatibility Model — CLIP Fine-Tuning

Trains a compatibility head on top of Fashion-CLIP embeddings using the Polyvore dataset.

**Dataset:** Polyvore Outfits (17,316 outfits, 70K+ items)

**Output:** `compatibility_head.pth` (~300 KB) — ready for Flask inference.

**Time:** ~30 min on T4 GPU (head only) / ~2 h for full fine-tune

## Setup

In [ ]:
!pip install torch torchvision transformers Pillow numpy scikit-learn tqdm gdown

In [ ]:
import json, random, os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE}")
BATCH_SIZE = 128
EMBED_DIM = 512  # CLIP ViT-B/32 text dim

## 1. Load Polyvore Dataset

Upload the JSON files from your `data/` folder to Colab or mount Google Drive.

If using Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
```

Then set `DATA_DIR = "/content/drive/MyDrive/polyvore"`

In [ ]:
# --- CONFIG ---
DATA_DIR = "/content/data"  # change to your path

def load_json(name):
    with open(f"{DATA_DIR}/{name}") as f:
        return json.load(f)

train = load_json("train_no_dup.json")
valid = load_json("valid_no_dup.json")
test  = load_json("test_no_dup.json")

print(f"Train: {len(train)} outfits")
print(f"Valid: {len(valid)} outfits")
print(f"Test:  {len(test)} outfits")

## 2. Compatibility Pairs

Each outfit is a positive example (all items are compatible).
Negative pairs come from items in different outfits.

In [ ]:
def build_pairs(outfits, neg_ratio=3):
    pos_pairs = []
    neg_pairs = []
    all_items = []
    
    for oi, outfit in enumerate(outfits):
        items = outfit["items"]
        names = [it["name"].strip().lower() for it in items]
        cat_ids = [it["categoryid"] for it in items]
        
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                pos_pairs.append((names[i], names[j], 1, cat_ids[i], cat_ids[j]))
        
        for it in items:
            all_items.append((it["name"].strip().lower(), it["categoryid"], oi))
    
    for _ in range(len(pos_pairs) * neg_ratio):
        a, b = random.sample(all_items, 2)
        if a[2] != b[2]:
            neg_pairs.append((a[0], b[0], 0, a[1], b[1]))
    
    print(f"Positive pairs: {len(pos_pairs)}")
    print(f"Negative pairs: {len(neg_pairs)}")
    return pos_pairs, neg_pairs

pos_train, neg_train = build_pairs(train, neg_ratio=3)
pos_valid, neg_valid = build_pairs(valid, neg_ratio=1)
pairs_train = pos_train + neg_train
pairs_valid = pos_valid + neg_valid
random.shuffle(pairs_train)
random.shuffle(pairs_valid)
print(f"\\nTrain pairs: {len(pairs_train)}")

## 3. CLIP Text Embeddings

Encode each unique item name with Fashion-CLIP's text encoder.

In [ ]:
from transformers import CLIPProcessor, CLIPModel

model_id = "patrickjohncyh/fashion-clip"
clip_model = CLIPModel.from_pretrained(model_id).to(DEVICE)
processor = CLIPProcessor.from_pretrained(model_id)
clip_model.eval()

print("CLIP model loaded")
print(f"Params: {sum(p.numel() for p in clip_model.parameters())/1e6:.1f}M")

In [ ]:
def get_text_embeddings(texts, batch_size=256):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Text embeddings"):
        batch = texts[i:i+batch_size]
        inputs = processor(text=batch, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        with torch.no_grad():
            emb = clip_model.get_text_features(**inputs)
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0)

all_names = []
for outfit in train + valid + test:
    for it in outfit["items"]:
        name = it["name"].strip().lower()
        if name not in all_names:
            all_names.append(name)

print(f"Unique item names: {len(all_names)}")
name_to_idx = {n: i for i, n in enumerate(all_names)}
print("Computing text embeddings...")
name_embeddings = get_text_embeddings(all_names)
print(f"Embeddings shape: {name_embeddings.shape}")

## 4. Dataset & DataLoader

In [ ]:
class CompatibilityDataset(Dataset):
    def __init__(self, pairs, emb_matrix, name_to_idx):
        self.data = []
        for name_a, name_b, label, cat_a, cat_b in pairs:
            if name_a in name_to_idx and name_b in name_to_idx:
                idx_a = name_to_idx[name_a]
                idx_b = name_to_idx[name_b]
                self.data.append((idx_a, idx_b, label, cat_a, cat_b))
        self.emb_matrix = emb_matrix
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        idx_a, idx_b, label, cat_a, cat_b = self.data[idx]
        emb_a = self.emb_matrix[idx_a]
        emb_b = self.emb_matrix[idx_b]
        return emb_a, emb_b, torch.tensor(label, dtype=torch.float32)

ds_train = CompatibilityDataset(pairs_train, name_embeddings, name_to_idx)
ds_valid = CompatibilityDataset(pairs_valid, name_embeddings, name_to_idx)
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
dl_valid = DataLoader(ds_valid, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train: {len(ds_train)} pairs, Valid: {len(ds_valid)} pairs")

## 5. Model: Compatibility Head

In [ ]:
class CompatibilityHead(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
    
    def forward(self, emb_a, emb_b):
        x = torch.cat([emb_a, emb_b], dim=1)
        return self.net(x).squeeze(-1)

model = CompatibilityHead().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print(f"Trainable params: {sum(p.numel() for p in model.parameters())}")
print(f"Model size: {sum(p.numel() for p in model.parameters())*4/1024:.0f} KB")

## 6. Training Loop (Head Only, ~30 min on T4)

In [ ]:
EPOCHS = 15
best_auc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for emb_a, emb_b, labels in tqdm(dl_train, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        emb_a, emb_b, labels = emb_a.to(DEVICE), emb_b.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(emb_a, emb_b)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for emb_a, emb_b, labels in dl_valid:
            emb_a, emb_b = emb_a.to(DEVICE), emb_b.to(DEVICE)
            logits = model(emb_a, emb_b)
            probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())
    
    preds = np.concatenate(all_preds)
    true = np.concatenate(all_labels)
    acc = accuracy_score(true, preds > 0.5)
    auc = roc_auc_score(true, preds)
    
    print(f"Loss: {train_loss/len(dl_train):.4f} | Acc: {acc:.3f} | AUC: {auc:.3f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "/content/compatibility_head.pth")
        print(f"  -> Saved (best AUC: {auc:.3f})")

print(f"\nBest AUC: {best_auc:.3f}")

## 7. Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load("/content/compatibility_head.pth"))
model.eval()

pos_test, neg_test = build_pairs(test, neg_ratio=1)
pairs_test = pos_test + neg_test
ds_test = CompatibilityDataset(pairs_test, name_embeddings, name_to_idx)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, num_workers=2)

all_preds, all_labels = [], []
with torch.no_grad():
    for emb_a, emb_b, labels in tqdm(dl_test, desc="Testing"):
        emb_a, emb_b = emb_a.to(DEVICE), emb_b.to(DEVICE)
        logits = model(emb_a, emb_b)
        probs = torch.sigmoid(logits)
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

preds = np.concatenate(all_preds)
true = np.concatenate(all_labels)
print(f"Test Accuracy: {accuracy_score(true, preds > 0.5):.3f}")
print(f"Test AUC:      {roc_auc_score(true, preds):.3f}")

## 8. Full Fine-Tune (Optional)

Unfreeze the last 6 CLIP transformer layers + head. Train end-to-end for better accuracy.

**Time:** ~2-4 h on T4. Supports checkpoint save/resume across Colab sessions.

In [ ]:
class FullCompatibilityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_id)
        self.head = CompatibilityHead()
        self._freeze_early_layers()

    def _freeze_early_layers(self):
        for name, param in self.clip.text_model.named_parameters():
            if "encoder.layers." in name:
                layer_num = int(name.split("encoder.layers.")[1].split(".")[0])
                if layer_num < 6:
                    param.requires_grad = False
            else:
                param.requires_grad = False
        for param in self.clip.vision_model.parameters():
            param.requires_grad = False
        self.clip.logit_scale.requires_grad_(False)

    def forward(self, text_a, text_b):
        emb_a = self.clip.get_text_features(**text_a)
        emb_b = self.clip.get_text_features(**text_b)
        return self.head(emb_a, emb_b)


class TextCompatibilityDataset(Dataset):
    def __init__(self, pairs):
        self.data = [(a, b, lbl) for a, b, lbl, _, _ in pairs]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_text(batch, processor=processor, device=DEVICE):
    texts_a = [item[0] for item in batch]
    texts_b = [item[1] for item in batch]
    labels = torch.tensor([item[2] for item in batch], dtype=torch.float32)
    tok_a = processor(text=texts_a, return_tensors="pt", padding=True, truncation=True).to(device)
    tok_b = processor(text=texts_b, return_tensors="pt", padding=True, truncation=True).to(device)
    return tok_a, tok_b, labels


dl_ft_train = DataLoader(
    TextCompatibilityDataset(pairs_train), batch_size=64, shuffle=True,
    collate_fn=collate_text, num_workers=2)
dl_ft_valid = DataLoader(
    TextCompatibilityDataset(pairs_valid), batch_size=64, shuffle=False,
    collate_fn=collate_text, num_workers=2)

full_model = FullCompatibilityModel().to(DEVICE)
ft_optimizer = torch.optim.AdamW(
    [p for p in full_model.parameters() if p.requires_grad],
    lr=1e-5, weight_decay=1e-5)
ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=10)

trainable = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in full_model.parameters() if not p.requires_grad)
print(f"Trainable: {trainable/1e6:.1f}M | Frozen: {frozen/1e6:.1f}M")
print(f"Expected time: ~2-4 h on T4")

## 8b. Full Fine-Tune Training Loop (with checkpoint resume)

In [ ]:
CHECKPOINT_PATH = "/content/ft_checkpoint.pt"
HEAD_PATH = "/content/compatibility_head.pth"
FT_EPOCHS = 10
start_epoch = 0
best_auc = 0.0

# --- Resume from checkpoint if exists ---
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    full_model.load_state_dict(ckpt["model_state_dict"])
    ft_optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    ft_scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_auc = ckpt.get("best_auc", 0.0)
    print(f"Resumed from epoch {ckpt['epoch']} (best AUC: {best_auc:.3f})")
else:
    print("No checkpoint found -- starting from scratch")

for epoch in range(start_epoch, FT_EPOCHS):
    full_model.train()
    train_loss = 0.0
    for tok_a, tok_b, labels in tqdm(dl_ft_train, desc=f"FT Epoch {epoch+1}/{FT_EPOCHS}"):
        labels = labels.to(DEVICE)
        ft_optimizer.zero_grad()
        logits = full_model(tok_a, tok_b)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(full_model.parameters(), 1.0)
        ft_optimizer.step()
        train_loss += loss.item()
    ft_scheduler.step()

    full_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for tok_a, tok_b, labels in dl_ft_valid:
            tok_a = {k: v.to(DEVICE) for k, v in tok_a.items()}
            tok_b = {k: v.to(DEVICE) for k, v in tok_b.items()}
            logits = full_model(tok_a, tok_b)
            probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())

    preds = np.concatenate(all_preds)
    true = np.concatenate(all_labels)
    acc = accuracy_score(true, preds > 0.5)
    auc = roc_auc_score(true, preds)

    print(f"Loss: {train_loss/len(dl_ft_train):.4f} | Acc: {acc:.3f} | AUC: {auc:.3f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(full_model.head.state_dict(), HEAD_PATH)
        print(f"  -> Saved head (best AUC: {auc:.3f})")

    # Save full checkpoint for resumption
    torch.save({
        "epoch": epoch,
        "model_state_dict": full_model.state_dict(),
        "optimizer_state_dict": ft_optimizer.state_dict(),
        "scheduler_state_dict": ft_scheduler.state_dict(),
        "best_auc": best_auc,
    }, CHECKPOINT_PATH)
    print(f"  Checkpoint saved (epoch {epoch+1}/{FT_EPOCHS})")

print(f"\nBest AUC: {best_auc:.3f}")
print(f"Head saved to: {HEAD_PATH}")
print(f"Full checkpoint saved to: {CHECKPOINT_PATH}")
print("Next session: re-run cells 1-5 + 20 + this cell to resume.")

## 9. Download & Deploy

Download the trained head and copy to your Flask server.

In [ ]:
from google.colab import files
files.download("/content/compatibility_head.pth")

---
## Deployment

Place `compatibility_head.pth` in `server/` and the Flask API will:
1. Load the head weights on startup
2. During generation, compute CLIP text embeddings for each item
3. Score each pair in an outfit through the head
4. Return outfits sorted by compatibility score

The model is already integrated via `server/compatibility_model.py`.
```python
from compatibility_model import OutfitScorer
scorer = OutfitScorer()
score = scorer.score_outfit(item_embeddings)  # higher = more compatible
```